# Assignment-4 COMP-5970 Jacob Murrah
## README
Design a small CNN from scratch (5–10 layers), train it on CIFAR-10, then
upgrade it with residual connections and compare. Finally, pretrain one of
your models with a self-supervised objective and fine-tune on labels. You’ll
analyze accuracy, data efficiency, convergence, and learned features.

## Dependencies
- **Python 3.x**
- **numpy**
- **matplotlib**
- **opencv-python**

*Note: If you are running this notebook in Google Colab, all the required packages are pre-installed.*

## Instructions
1. **Run All Cells:** Please click on \"Runtime\" > \"Run all\" to execute the entire notebook sequentially.
2. **Review the Outputs:** The notebook is organized into several sections. Ensure that all cells run without errors.

## GenAI Usage Declaration
a

## Statement of Contribution
This was an individual project and I did everything in it.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import cv2
import torch, torch.nn as nn, torch.nn.functional as F
import tensorflow as tf
import torchvision, torchvision.transforms as transforms
from torch.utils.data import Subset

2025-11-21 17:39:06.016455: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-21 17:39:06.056354: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-11-21 17:39:06.717863: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [3]:
# Load CIFAR-10 dataset
# Note: CIFAR-10 has 10 classes
    # 1) airplane
    # 2) automobile
    # 3) bird
    # 4) cat
    # 5) deer
    # 6) dog
    # 7) frog
    # 8) horse
    # 9) ship
    # 10) truck

train_transform = transforms.Compose([
    transforms.Resize((32,32)),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.4914, 0.4822, 0.4465], # known for CIFAR-10
        std=[0.2023, 0.1994, 0.2010] # known for CIFAR-10
    )
])

test_transform = transforms.Compose([
    transforms.Resize((32,32)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.4914, 0.4822, 0.4465], 
        std=[0.2023, 0.1994, 0.2010]
    )
])

train_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=True, transform=train_transform, download=True
)
test_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=False, transform=test_transform, download=True
)

train_loader = torch.utils.data.DataLoader(
    dataset=train_dataset, batch_size=96, num_workers=2, shuffle=True
)
test_loader = torch.utils.data.DataLoader(
    dataset=test_dataset, batch_size=96, shuffle=True
)

In [4]:
class CIFARClassifier(nn.Module):
    def __init__(
        self,
        encoder,
        latent_ch=128,
        num_classes=10,
        in_channels=3,
    ):
        super().__init__()
        self.encoder = encoder
        self.head = nn.Sequential(
            nn.Linear(latent_ch, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.head(z)

def train_epoch(m, criterion, opt):
    m.train()
    total, correct, loss_sum = 0, 0, 0.0
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        opt.zero_grad()

        logits = m(images)
        loss = criterion(logits, labels)

        loss.backward()
        opt.step()

        pred = torch.argmax(logits, dim=1)
        correct += (pred == labels).sum().item()
        loss_sum += loss.item() * labels.size(0)

        total += labels.size(0)

    return loss_sum / total, correct / total

def test_epoch(m, criterion):
    m.eval()
    total, correct, loss_sum = 0, 0, 0.0
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)

            logits = m(images)
            loss = criterion(logits, labels)

            pred = torch.argmax(logits, dim=1)
            correct += (pred == labels).sum().item()
            loss_sum += loss.item() * labels.size(0)

            total += labels.size(0)

    return loss_sum / total, correct / total

def train_model(m, c, o):
    param_count = sum(p.numel() for p in m.parameters())
    print(f"Total parameters: {param_count}")

    for epoch in range(30):
        tr_loss, tr_acc = train_epoch(m, c, o)
        te_loss, te_acc = test_epoch(m, c)
        print(
            f"Epoch {epoch+1}: train loss {tr_loss:.4f}, acc {tr_acc:.3f} | test loss {te_loss:.4f}, acc {te_acc:.3f}"
        )


# Task 1) Small CNN (from scratch)
Implement and train a compact CNN classifier on CIFAR-10 with the
following specifications:
- 5–10 learnable layers (conv/normalization/activation/linear count
toward depth; pooling doesn’t).
- ≤ 1.5M parameters (show a param count).
- Use standard training (cross-entropy) and reasonable augments
(random crop/flip, optional color jitter).
- Train to reasonable test accuracy (don’t overtrain; ~20–30 epochs or
early stopping is fine).

## Description of Approach
a

In [5]:
class SimpleCNNEncoder(nn.Module):
    def __init__(self, in_channels=3, latent_ch=128):
        super().__init__()
        self.dropout = nn.Dropout(0.5)

        self.conv1 = nn.Conv2d(
            in_channels=in_channels,
            out_channels=64,
            kernel_size=3,
            padding=1,
            bias=False
        )
        self.bn1 = nn.BatchNorm2d(64)
        self.relu1 = nn.ReLU(inplace=True)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.conv2 = nn.Conv2d(
            in_channels=64,
            out_channels=128,
            kernel_size=3,
            padding=1,
            bias=False
        )
        self.bn2 = nn.BatchNorm2d(128)
        self.relu2 = nn.ReLU(inplace=True)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.conv3 = nn.Conv2d(
            in_channels=128,
            out_channels=256,
            kernel_size=3,
            padding=1,
            bias=False
        )
        self.bn3 = nn.BatchNorm2d(256)
        self.relu3 = nn.ReLU(inplace=True)
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.fc = nn.Linear(
            in_features=256 * 4 * 4, out_features=latent_ch
        )

    def forward(self, x):
        x = self.pool1(self.relu1(self.bn1(self.conv1(x))))
        x = self.pool2(self.relu2(self.bn2(self.conv2(x))))
        x = self.pool3(self.relu3(self.bn3(self.conv3(x))))
        x = x.view(x.size(0), -1)
        x = self.dropout(x)
        x = self.fc(x)
        return x

In [6]:
model1 = CIFARClassifier(encoder=SimpleCNNEncoder()).to(device)
opt1 = torch.optim.SGD(
    model1.parameters(), lr=0.001, weight_decay=1e-4, momentum=0.9
)
criterion1 = nn.CrossEntropyLoss()

# train_model(model1, criterion1, opt1)

# Task 2) Improve your CNN
Modify your CNN to include identity (skip) connections where shapes
match (e.g., two convs per block), and projection when they don’t (1×1 conv).
Make sure to follow the below specifications:
- Keep overall depth and parameter budget comparable (±20% is fine).
- Reuse the same optimizer/augments unless you justify changes.
- Train and compare against Part A.

## Description of Approach
a

In [7]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        
        self.conv = nn.Conv2d(
            in_channels, out_channels, kernel_size=3, padding=1, bias=False
        )
        self.bn = nn.BatchNorm2d(out_channels)
        
        if in_channels != out_channels:
            self.proj = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
            self.proj_bn = nn.BatchNorm2d(out_channels)
        else:
            self.proj = None
        
        self.relu = nn.ReLU(inplace=True)
    
    def forward(self, x):
        identity = x

        out = self.conv(x)
        out = self.bn(out)

        if self.proj is not None:
            identity = self.proj(identity)
            identity = self.proj_bn(identity)

        out += identity
        out = self.relu(out)
        
        return out


class SkipCNNEncoder(nn.Module):
    def __init__(self, in_channels=3, latent_ch=128):
        super().__init__()
        self.dropout = nn.Dropout(0.5)

        self.block1 = ResidualBlock(in_channels, 64)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.block2 = ResidualBlock(64, 128)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.block3 = ResidualBlock(128, 256)
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.fc = nn.Linear(
            in_features=256 * 4 * 4, out_features=latent_ch
        )

    def forward(self, x):
        x = self.pool1(self.block1(x))
        x = self.pool2(self.block2(x))
        x = self.pool3(self.block3(x))
        
        x = x.view(x.size(0), -1)
        x = self.dropout(x)
        x = self.fc(x)
        return x

In [8]:
model2 = CIFARClassifier(encoder=SkipCNNEncoder()).to(device)
opt2 = torch.optim.SGD(
    model2.parameters(), lr=0.001, weight_decay=1e-4, momentum=0.9
)
criterion2 = nn.CrossEntropyLoss()

# train_model(model2, criterion2, opt2)

# Task 2) Self-Supervised Pretraining + Fine-tuning
Pretrain either your vanilla CNN or residual CNN without labels, then
fine-tune on labeled CIFAR-10, with the following specifications:
- Choose one SSL strategy from the below list:
  * Predictive: Rotation prediction (0/90/180/270) or patch jigsaw.
  * Contrastive: SimCLR-style instance discrimination (two strong
views).
  * Masked modeling / reconstruction: Mask random patches and
predict pixels/features.
- Follow the below training protocol:
  * Pretrain on the entire unlabeled CIFAR-10 train set (labels
hidden).
  * Evaluation B (Fine-tune): Unfreeze and fine-tune end-to-end
with 10% labeled data (make sure to sample uniformly across
classes!).

## Description of Approach
a

In [9]:
def create_rotated_batch(images, device):
    """
    Takes a batch of images, creates 4 rotated copies of each,
    and generates labels 0, 1, 2, 3.
    """
    batch_size = images.shape[0]
    
    # 1. Create the 4 rotations (0, 90, 180, 270)
    x0 = images
    x90 = torch.rot90(images, 1, [2, 3])
    x180 = torch.rot90(images, 2, [2, 3])
    x270 = torch.rot90(images, 3, [2, 3])
    
    # 2. Concatenate into a batch 4x the size
    rot_images = torch.cat([x0, x90, x180, x270], dim=0)
    
    # 3. Create labels (0=0deg, 1=90deg, 2=180deg, 3=270deg)
    y0 = torch.zeros(batch_size, dtype=torch.long, device=device)
    y90 = torch.ones(batch_size, dtype=torch.long, device=device)
    y180 = torch.full((batch_size,), 2, dtype=torch.long, device=device)
    y270 = torch.full((batch_size,), 3, dtype=torch.long, device=device)
    
    rot_labels = torch.cat([y0, y90, y180, y270], dim=0)
    
    # Shuffle to prevent the model from learning based on batch order
    idx = torch.randperm(rot_images.size(0))
    return rot_images[idx], rot_labels[idx]

def pretrain_epoch(m, criterion, opt):
    """
    Modified training loop that IGNORES real labels and uses 
    generated rotation labels.
    """
    m.train()
    total_loss = 0.0
    total_acc = 0.0
    total_samples = 0
    
    for images, _ in train_loader: # Note: We ignore the real labels '_'
        images = images.to(device)
        
        # Generate the self-supervised task
        rot_images, rot_labels = create_rotated_batch(images, device)
        
        opt.zero_grad()
        logits = m(rot_images)
        loss = criterion(logits, rot_labels)
        
        loss.backward()
        opt.step()
        
        # Metrics
        pred = torch.argmax(logits, dim=1)
        total_acc += (pred == rot_labels).sum().item()
        total_loss += loss.item() * rot_images.size(0)
        total_samples += rot_images.size(0)
        
    return total_loss / total_samples, total_acc / total_samples

def get_balanced_10_percent_subset(dataset, num_classes=10):
    """
    Selects exactly 10% of data, ensuring equal count per class.
    """
    targets = np.array(dataset.targets)
    indices = []
    for c in range(num_classes):
        # Get all indices for this class
        class_indices = np.where(targets == c)[0]
        # Pick 10%
        n_samples = int(len(class_indices) * 0.10)
        selected = np.random.choice(class_indices, n_samples, replace=False)
        indices.extend(selected)
    
    return Subset(dataset, indices)

In [10]:
# ==========================================
# PHASE 1: Self-Supervised Pretraining (SSL)
# ==========================================
print("--- Starting Phase 1: Rotational Pretraining ---")

# 1. Initialize Model with 4 output classes (0, 90, 180, 270)
ssl_model = CIFARClassifier(encoder=SimpleCNNEncoder(), num_classes=4).to(device)

# 2. Standard Optimizer
ssl_opt = torch.optim.SGD(
    ssl_model.parameters(), lr=0.001, weight_decay=1e-4, momentum=0.9
)
ssl_criterion = nn.CrossEntropyLoss()

# 3. Train on Rotation Task (using the FULL dataset)
for epoch in range(15): # 10-15 epochs is usually enough for SSL
    loss, acc = pretrain_epoch(ssl_model, ssl_criterion, ssl_opt)
    print(f"SSL Epoch {epoch+1}: Rotation Loss {loss:.4f}, Rotation Acc {acc:.3f}")

print("--- Pretraining Complete ---")


# ==========================================
# PHASE 2: Fine-tuning on 10% Labeled Data
# ==========================================
print("\n--- Starting Phase 2: Fine-tuning on 10% Data ---")

# 1. Modify the Head
# We keep the trained encoder, but throw away the 4-class head 
# and replace it with a new 10-class head for CIFAR objects.
ssl_model.head[2] = nn.Linear(64, 10).to(device) 

# 2. Create the 10% Data Subset
subset_dataset = get_balanced_10_percent_subset(train_dataset)
finetune_loader = torch.utils.data.DataLoader(
    subset_dataset, batch_size=96, shuffle=True, num_workers=2
)

# 3. Re-initialize Optimizer 
# We must do this because we added a new layer (head[2])
finetune_opt = torch.optim.SGD(
    ssl_model.parameters(), lr=0.001, weight_decay=1e-4, momentum=0.9
)
finetune_criterion = nn.CrossEntropyLoss()

# 4. Standard Training Loop (using your original functions)
# Note: We use finetune_loader here, but test_loader (full) for evaluation
for epoch in range(30):
    # We manually call train_epoch but pass the specialized loader
    ssl_model.train()
    total, correct, loss_sum = 0, 0, 0.0
    
    for images, labels in finetune_loader: # <--- USING 10% LOADER
        images = images.to(device)
        labels = labels.to(device)

        finetune_opt.zero_grad()
        logits = ssl_model(images)
        loss = finetune_criterion(logits, labels)

        loss.backward()
        finetune_opt.step()

        pred = torch.argmax(logits, dim=1)
        correct += (pred == labels).sum().item()
        loss_sum += loss.item() * labels.size(0)
        total += labels.size(0)
    
    tr_loss = loss_sum / total
    tr_acc = correct / total
    
    # Test on FULL test set
    te_loss, te_acc = test_epoch(ssl_model, finetune_criterion)
    
    print(f"Fine-tune Epoch {epoch+1}: Train Loss {tr_loss:.4f} Acc {tr_acc:.3f} | Test Acc {te_acc:.3f}")

--- Starting Phase 1: Rotational Pretraining ---
SSL Epoch 1: Rotation Loss 1.1928, Rotation Acc 0.483
SSL Epoch 2: Rotation Loss 1.0083, Rotation Acc 0.588
SSL Epoch 3: Rotation Loss 0.9307, Rotation Acc 0.620
SSL Epoch 4: Rotation Loss 0.8823, Rotation Acc 0.640
SSL Epoch 5: Rotation Loss 0.8444, Rotation Acc 0.658
SSL Epoch 6: Rotation Loss 0.8173, Rotation Acc 0.669
SSL Epoch 7: Rotation Loss 0.7927, Rotation Acc 0.680
SSL Epoch 8: Rotation Loss 0.7757, Rotation Acc 0.687
SSL Epoch 9: Rotation Loss 0.7587, Rotation Acc 0.695
SSL Epoch 10: Rotation Loss 0.7467, Rotation Acc 0.700
SSL Epoch 11: Rotation Loss 0.7338, Rotation Acc 0.705
SSL Epoch 12: Rotation Loss 0.7213, Rotation Acc 0.712
SSL Epoch 13: Rotation Loss 0.7102, Rotation Acc 0.717
SSL Epoch 14: Rotation Loss 0.6977, Rotation Acc 0.723
SSL Epoch 15: Rotation Loss 0.6893, Rotation Acc 0.726
--- Pretraining Complete ---

--- Starting Phase 2: Fine-tuning on 10% Data ---
Fine-tune Epoch 1: Train Loss 2.1493 Acc 0.201 | Test A

In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T

# ============================================================
# HELPER CLASSES (Loss, Transform Wrapper, Model Wrapper)
# ============================================================

# --- 1. The Contrastive Loss Function (NT-Xent) ---
class NTXentLoss(nn.Module):
    def __init__(self, temperature=0.5):
        super().__init__()
        self.temperature = temperature

    def forward(self, z_i, z_j):
        batch_size = z_i.shape[0]
        
        # Concatenate the two views: [Batch, Dim] -> [2*Batch, Dim]
        features = torch.cat([z_i, z_j], dim=0)
        
        # Calculate Cosine Similarity Matrix
        # (a . b) / (|a| * |b|)
        features = F.normalize(features, dim=1)
        similarity_matrix = torch.matmul(features, features.T)
        
        # Create the labels (matches are offset by batch_size)
        # If batch is 3, labels for [0,1,2] are [3,4,5]
        labels = torch.cat([
            torch.arange(batch_size) + batch_size, 
            torch.arange(batch_size)
        ], dim=0).to(features.device)
        
        # Mask out self-similarity (diagonal)
        mask = torch.eye(labels.shape[0], dtype=torch.bool).to(features.device)
        # We remove diagonals by setting them to very small value so softmax ignores them
        similarity_matrix.masked_fill_(mask, -9e15)
        
        # Compute Cross Entropy
        logits = similarity_matrix / self.temperature
        loss = F.cross_entropy(logits, labels)
        return loss

# --- 2. The SimCLR Augmentation Wrapper ---
class SimCLRTransform:
    """
    This applies the transform TWICE to get two different views of the same image.
    """
    def __init__(self, transform):
        self.transform = transform

    def __call__(self, x):
        return self.transform(x), self.transform(x)

# --- 3. The SimCLR Model Wrapper ---
class SimCLR(nn.Module):
    def __init__(self, encoder, latent_dim=128, proj_dim=64):
        super().__init__()
        self.encoder = encoder
        
        # The Projection Head (MLP)
        # Important: We train with this, but throw it away later!
        self.projector = nn.Sequential(
            nn.Linear(latent_dim, latent_dim),
            nn.ReLU(),
            nn.Linear(latent_dim, proj_dim)
        )

    def forward(self, x):
        h = self.encoder(x) # Get features (128 dim)
        z = self.projector(h) # Project to embedding space (64 dim)
        return h, z

# ============================================================
# PHASE 1: SimCLR Pretraining
# ============================================================
print("--- Starting Phase 1: SimCLR Pretraining ---")

# 1. Define Strong Augmentations (Crucial for SimCLR!)
# We define these separately because SimCLR needs RandomResizedCrop (Zooming)
# which is too aggressive for standard supervised training.
simclr_augments = T.Compose([
    T.RandomResizedCrop(32, scale=(0.2, 1.0)),
    T.RandomHorizontalFlip(),
    T.ColorJitter(0.4, 0.4, 0.4, 0.1), # (brightness, contrast, saturation, hue)
    T.RandomGrayscale(p=0.1),
    T.ToTensor(),
    T.Normalize(mean=[0.4914, 0.4822, 0.4465], std=[0.2023, 0.1994, 0.2010])
])

# 2. Create Dataset with Dual-View Transform
# We create a fresh dataset instance for SimCLR to apply the dual-view transform
simclr_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=True, 
    transform=SimCLRTransform(simclr_augments), # <--- Returns (img1, img2)
    download=True
)

simclr_loader = torch.utils.data.DataLoader(
    simclr_dataset, batch_size=96, shuffle=True, num_workers=2, drop_last=True
)

# 3. Initialize Model
# We use your SimpleCNNEncoder
encoder = SimpleCNNEncoder(latent_ch=128) 
simclr_model = SimCLR(encoder, latent_dim=128, proj_dim=64).to(device)

# 4. Optimizer & Loss
# SimCLR likes higher LR, but we'll stick to your standard for stability
optimizer = torch.optim.SGD(simclr_model.parameters(), lr=0.001, momentum=0.9, weight_decay=1e-4)
criterion = NTXentLoss(temperature=0.5)

# 5. Pretraining Loop
for epoch in range(15): # 15-20 epochs recommended
    simclr_model.train()
    total_loss = 0
    
    for (img1, img2), _ in simclr_loader: # Ignore labels (_)
        img1, img2 = img1.to(device), img2.to(device)
        
        optimizer.zero_grad()
        
        # Forward pass for both views
        _, z1 = simclr_model(img1) # We only need z (projection) for loss
        _, z2 = simclr_model(img2)
        
        loss = criterion(z1, z2)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
    print(f"SimCLR Epoch {epoch+1}: Loss {total_loss/len(simclr_loader):.4f}")

print("--- SimCLR Pretraining Complete ---")


# ============================================================
# PHASE 2: Fine-tuning (Linear Probing -> Fine Tuning)
# ============================================================
print("\n--- Starting Phase 2: Fine-tuning on 10% Data ---")

# 1. CONVERT MODEL FOR CLASSIFICATION
# We extract the trained encoder and wrap it in your Classifier class
trained_encoder = simclr_model.encoder
classifier = CIFARClassifier(encoder=trained_encoder, num_classes=10).to(device)

# 2. FREEZE ENCODER (Linear Probing)
for param in classifier.encoder.parameters():
    param.requires_grad = False

# 3. DATA SETUP
# NOTE: We use 'train_dataset' which you updated globally in the previous step.
# This dataset ALREADY contains the ColorJitter/Flip transforms.
subset_dataset = get_balanced_10_percent_subset(train_dataset) 

finetune_loader = torch.utils.data.DataLoader(
    subset_dataset, batch_size=96, shuffle=True, num_workers=2
)

# Optimizer for Head Only
head_opt = torch.optim.SGD(classifier.head.parameters(), lr=0.01, momentum=0.9)
cls_criterion = nn.CrossEntropyLoss()

# --- Step A: Warmup Head ---
print("--- Step A: Linear Probing (Head Only) ---")
for epoch in range(5):
    classifier.train()
    for images, labels in finetune_loader:
        images, labels = images.to(device), labels.to(device)
        head_opt.zero_grad()
        logits = classifier(images)
        loss = cls_criterion(logits, labels)
        loss.backward()
        head_opt.step()
    print(f"Warmup Epoch {epoch+1} Done")

# --- Step B: Full Fine-Tuning ---
print("--- Step B: Full Fine-Tuning (Unfrozen) ---")

# Unfreeze Encoder
for param in classifier.encoder.parameters():
    param.requires_grad = True

# Optimizer for Everything
full_opt = torch.optim.SGD(classifier.parameters(), lr=0.001, momentum=0.9)

# Final Loop
for epoch in range(30):
    tr_loss, tr_acc = train_epoch(classifier, cls_criterion, full_opt) # Helper from Part 1
    
    # Evaluate on FULL test set
    te_loss, te_acc = test_epoch(classifier, cls_criterion) # Helper from Part 1
    
    print(f"Fine-tune Epoch {epoch+1}: Train Loss {tr_loss:.4f} Acc {tr_acc:.3f} | Test Acc {te_acc:.3f}")

--- Starting Phase 1: SimCLR Pretraining ---
SimCLR Epoch 1: Loss 4.4704
SimCLR Epoch 2: Loss 4.2986
SimCLR Epoch 3: Loss 4.2484
SimCLR Epoch 4: Loss 4.2171
SimCLR Epoch 5: Loss 4.1810
SimCLR Epoch 6: Loss 4.1534
SimCLR Epoch 7: Loss 4.1355
SimCLR Epoch 8: Loss 4.1205
SimCLR Epoch 9: Loss 4.1044
SimCLR Epoch 10: Loss 4.0954
SimCLR Epoch 11: Loss 4.0816
SimCLR Epoch 12: Loss 4.0701
SimCLR Epoch 13: Loss 4.0645
SimCLR Epoch 14: Loss 4.0591
SimCLR Epoch 15: Loss 4.0509
--- SimCLR Pretraining Complete ---

--- Starting Phase 2: Fine-tuning on 10% Data ---
--- Step A: Linear Probing (Head Only) ---
Warmup Epoch 1 Done
Warmup Epoch 2 Done
Warmup Epoch 3 Done
Warmup Epoch 4 Done
Warmup Epoch 5 Done
--- Step B: Full Fine-Tuning (Unfrozen) ---
Fine-tune Epoch 1: Train Loss 1.3423 Acc 0.516 | Test Acc 0.593
Fine-tune Epoch 2: Train Loss 1.1672 Acc 0.580 | Test Acc 0.628
Fine-tune Epoch 3: Train Loss 1.0731 Acc 0.617 | Test Acc 0.651
Fine-tune Epoch 4: Train Loss 1.0074 Acc 0.643 | Test Acc 0.684

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T

# ==========================================
# 1. THE MASKING FUNCTION
# ==========================================
def apply_random_mask(images, mask_size=12):
    """
    Replaces a random square patch in the image with zeros (black).
    CIFAR images are 32x32. We mask a 12x12 box.
    """
    B, C, H, W = images.shape
    masked_images = images.clone()
    
    # We generate random top-left corners for the mask
    # Limit range so mask stays inside image
    top = torch.randint(0, H - mask_size, (B,))
    left = torch.randint(0, W - mask_size, (B,))
    
    for i in range(B):
        t, l = top[i], left[i]
        masked_images[i, :, t:t+mask_size, l:l+mask_size] = 0.0
        
    return masked_images

# ==========================================
# 2. THE DECODER (Inverse of your Encoder)
# ==========================================
class SimpleCNNDecoder(nn.Module):
    def __init__(self, latent_ch=128):
        super().__init__()
        # 1. Expand latent vector back to feature map size (256 channels x 4x4)
        self.fc_expand = nn.Linear(latent_ch, 256 * 4 * 4)
        
        # 2. Upsampling Layers (Mirroring the Encoder's pooling)
        # 4x4 -> 8x8
        self.deconv1 = nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1)
        self.bn1 = nn.BatchNorm2d(128)
        
        # 8x8 -> 16x16
        self.deconv2 = nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        
        # 16x16 -> 32x32
        self.deconv3 = nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1)
        self.bn3 = nn.BatchNorm2d(32)
        
        # Final projection to 3 RGB channels
        self.final_conv = nn.Conv2d(32, 3, kernel_size=3, padding=1)
        
    def forward(self, z):
        # Unflatten: (Batch, 128) -> (Batch, 256, 4, 4)
        x = self.fc_expand(z)
        x = x.view(-1, 256, 4, 4)
        
        x = F.relu(self.bn1(self.deconv1(x))) # -> 8x8
        x = F.relu(self.bn2(self.deconv2(x))) # -> 16x16
        x = F.relu(self.bn3(self.deconv3(x))) # -> 32x32
        
        # Final layer usually sigmoid (if 0-1) or nothing (if normalized)
        # Since our inputs are normalized, we output raw logits
        x = self.final_conv(x) 
        return x

# ==========================================
# 3. THE AUTOENCODER WRAPPER
# ==========================================
class MaskedAutoencoder(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = SimpleCNNDecoder(latent_ch=128)
        
    def forward(self, x):
        z = self.encoder(x)
        reconstruction = self.decoder(z)
        return reconstruction

# ============================================================
# PHASE 1: Masked Pretraining (Reconstruction)
# ============================================================
print("--- Starting Phase 1: Masked Reconstruction Pretraining ---")

# 1. Setup Model
encoder = SimpleCNNEncoder(latent_ch=128)
mae_model = MaskedAutoencoder(encoder).to(device)

# 2. Optimization
# MSE Loss compares predicted pixels vs real pixels
mae_criterion = nn.MSELoss()
optimizer = torch.optim.SGD(mae_model.parameters(), lr=0.01, momentum=0.9, weight_decay=1e-4)

# 3. Train Loop
for epoch in range(15):
    mae_model.train()
    total_loss = 0
    
    for images, _ in train_loader: # Ignore labels
        images = images.to(device)
        
        # Create Masked Version
        masked_images = apply_random_mask(images, mask_size=10)
        
        optimizer.zero_grad()
        
        # Forward pass: Encoder sees masked image -> Decoder predicts ORIGINAL image
        reconstruction = mae_model(masked_images)
        
        # Loss: Compare Reconstruction to ORIGINAL (Unmasked) image
        loss = mae_criterion(reconstruction, images)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
    # Loss here is Mean Squared Error (Pixel difference)
    print(f"MAE Epoch {epoch+1}: Reconstruction MSE Loss {total_loss/len(train_loader):.4f}")

print("--- MAE Pretraining Complete ---")

# ============================================================
# PHASE 2: Fine-tuning on 10% Data
# ============================================================
print("\n--- Starting Phase 2: Fine-tuning on 10% Data ---")

# 1. Extract Encoder & Attach Classifier
# We throw away the decoder now.
trained_encoder = mae_model.encoder
classifier = CIFARClassifier(encoder=trained_encoder, num_classes=10).to(device)

# 2. FREEZE ENCODER (Warmup)
for param in classifier.encoder.parameters():
    param.requires_grad = False

# 3. Dataset Setup (10%)
subset_dataset = get_balanced_10_percent_subset(train_dataset)
finetune_loader = torch.utils.data.DataLoader(
    subset_dataset, batch_size=96, shuffle=True, num_workers=2
)

# Optimizer for Head
head_opt = torch.optim.SGD(classifier.head.parameters(), lr=0.01, momentum=0.9)
cls_criterion = nn.CrossEntropyLoss()

# --- Step A: Warmup Head ---
print("--- Step A: Linear Probing (Head Only) ---")
for epoch in range(5):
    classifier.train()
    for images, labels in finetune_loader:
        images, labels = images.to(device), labels.to(device)
        head_opt.zero_grad()
        logits = classifier(images)
        loss = cls_criterion(logits, labels)
        loss.backward()
        head_opt.step()
    print(f"Warmup Epoch {epoch+1} Done")

# --- Step B: Full Fine-Tuning ---
print("--- Step B: Full Fine-Tuning (Unfrozen) ---")

# Unfreeze
for param in classifier.encoder.parameters():
    param.requires_grad = True

# Full Optimizer
full_opt = torch.optim.SGD(classifier.parameters(), lr=0.001, momentum=0.9)

# Final Loop
for epoch in range(30):
    tr_loss, tr_acc = train_epoch(classifier, cls_criterion, full_opt)
    te_loss, te_acc = test_epoch(classifier, cls_criterion)
    
    print(f"Fine-tune Epoch {epoch+1}: Train Loss {tr_loss:.4f} Acc {tr_acc:.3f} | Test Acc {te_acc:.3f}")

--- Starting Phase 1: Masked Reconstruction Pretraining ---
MAE Epoch 1: Reconstruction MSE Loss 0.6855
MAE Epoch 2: Reconstruction MSE Loss 0.4700
MAE Epoch 3: Reconstruction MSE Loss 0.4248
MAE Epoch 4: Reconstruction MSE Loss 0.4007
MAE Epoch 5: Reconstruction MSE Loss 0.3815
MAE Epoch 6: Reconstruction MSE Loss 0.3611
MAE Epoch 7: Reconstruction MSE Loss 0.3512
MAE Epoch 8: Reconstruction MSE Loss 0.3445
MAE Epoch 9: Reconstruction MSE Loss 0.3388
MAE Epoch 10: Reconstruction MSE Loss 0.3274
MAE Epoch 11: Reconstruction MSE Loss 0.3240
MAE Epoch 12: Reconstruction MSE Loss 0.3207
MAE Epoch 13: Reconstruction MSE Loss 0.3149
MAE Epoch 14: Reconstruction MSE Loss 0.3095
MAE Epoch 15: Reconstruction MSE Loss 0.3058
--- MAE Pretraining Complete ---

--- Starting Phase 2: Fine-tuning on 10% Data ---
--- Step A: Linear Probing (Head Only) ---
Warmup Epoch 1 Done
Warmup Epoch 2 Done
Warmup Epoch 3 Done
Warmup Epoch 4 Done
Warmup Epoch 5 Done
--- Step B: Full Fine-Tuning (Unfrozen) ---
Fin